In [7]:
import os
import json
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import CubicSpline

# ---------- 設定 ----------
json_path = "./testdistance_estimates.json"
output_dir = "./scene_spline_fixed"
os.makedirs(output_dir, exist_ok=True)

# ---------- ジャンプ補正 ----------
def suppress_jumps(data, threshold=10.0):
    corrected = data.copy()
    for i in range(1, len(corrected)):
        if abs(corrected[i] - corrected[i - 1]) > threshold:
            corrected[i] = corrected[i - 1]
    return corrected

# ---------- 安定区間抽出 ----------
def find_stable_segments(data, diff_threshold=3.0, min_length=10, min_value=15.0):
    diffs = np.abs(np.diff(data, prepend=data[0]))
    stable_mask = (diffs < diff_threshold) & (data > min_value)

    segments = []
    start = None
    for i, val in enumerate(stable_mask):
        if val:
            if start is None:
                start = i
        else:
            if start is not None and i - start >= min_length:
                segments.append((start, i - 1))
            start = None
    if start is not None and len(data) - start >= min_length:
        segments.append((start, len(data) - 1))
    return segments

# ---------- スプライン補完（代表点のyを中央値±範囲に制限） ----------
def apply_spline_fit_partial(x_all, data, stable_segments, clip_margin=10.0):
    stable_x = []
    stable_y = []

    median_val = np.median(data)
    min_y = median_val - clip_margin
    max_y = median_val + clip_margin

    for start, end in stable_segments:
        for i in range(start, end + 1):
            if min_y <= data[i] <= max_y:
                stable_x.append(i)
                stable_y.append(data[i])

    if len(stable_x) < 4:
        return data

    unique_pairs = list({x: y for x, y in zip(stable_x, stable_y)}.items())
    if len(unique_pairs) < 4:
        return data

    unique_pairs.sort()
    sorted_x, sorted_y = zip(*unique_pairs)
    sorted_x = np.array(sorted_x)
    sorted_y = np.clip(np.array(sorted_y), min_y, max_y)

    spline = CubicSpline(sorted_x, sorted_y, bc_type='natural')

    smoothed = data.copy()
    for i in range(len(data)):
        if sorted_x[0] <= i <= sorted_x[-1]:
            smoothed[i] = float(np.clip(spline(i), min_y, max_y))
    return smoothed

# ---------- メイン処理 ----------
if not os.path.exists(json_path):
    print(f"❌ JSONファイルが見つかりません: {json_path}")
else:
    with open(json_path, encoding="utf-8") as f:
        data = json.load(f)

    for scene_id, frame_data in data.items():
        frame_keys = sorted(frame_data.keys(), key=lambda x: int(x.replace("frame_", "")))
        distances = np.array([frame_data[k] for k in frame_keys], dtype=float)

        jump_corrected = suppress_jumps(distances)
        segments = find_stable_segments(jump_corrected, diff_threshold=3.0, min_length=10, min_value=15.0)
        x_all = np.arange(len(distances))
        smoothed = apply_spline_fit_partial(x_all, jump_corrected, segments, clip_margin=10.0)

        plt.figure(figsize=(10, 6))
        plt.plot(distances, label="Original", alpha=0.4, marker='o', markersize=3)
        plt.plot(jump_corrected, label="Jump-corrected", linestyle='--', marker='x', markersize=3)
        plt.plot(smoothed, label="Spline Smoothed", linewidth=2, color="green")
        plt.title(f"Scene {scene_id} - Spline Fit with Anchors (Partial)")
        plt.xlabel("Frame Index")
        plt.ylabel("Distance (m)")
        plt.grid(True)
        plt.legend()
        plt.tight_layout()

        save_path = os.path.join(output_dir, f"{scene_id}_spline_fixed.png")
        plt.savefig(save_path)
        plt.close()

    print("✅ 全シーンにスプライン補完（極端な盛り上がり除去）を実施しました。")


✅ 全シーンにスプライン補完（極端な盛り上がり除去）を実施しました。


In [11]:
import os
import json
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import RANSACRegressor, LinearRegression

# ---------- 設定 ----------
json_path = "./testdistance_estimates.json"
scene_id = "047"
output_path = f"./scene211_ransac_fixed.png"

# ---------- RANSAC 補間 ----------
def ransac_fit(data, min_valid=10.0, max_valid=100.0):
    x = np.arange(len(data)).reshape(-1, 1)
    y = np.array(data)

    valid_mask = (y >= min_valid) & (y <= max_valid)
    x_valid = x[valid_mask]
    y_valid = y[valid_mask]

    if len(x_valid) < 2:
        return data

    model = RANSACRegressor(estimator=LinearRegression(), min_samples=5, residual_threshold=3.0)
    model.fit(x_valid, y_valid)
    y_pred = model.predict(x)

    return y_pred

# ---------- 実行 ----------
with open(json_path, encoding="utf-8") as f:
    all_data = json.load(f)

frame_data = all_data[scene_id]
frame_keys = sorted(frame_data.keys(), key=lambda x: int(x.replace("frame_", "")))
distances = np.array([frame_data[k] for k in frame_keys], dtype=float)

smoothed = ransac_fit(distances, min_valid=10.0)

# ---------- 描画 ----------
plt.figure(figsize=(10, 6))
plt.plot(distances, label="Original", alpha=0.4, marker='o', markersize=3)
plt.plot(smoothed, label="RANSAC Interpolated", linewidth=2, color="green")
plt.title(f"Scene {scene_id} - RANSAC Linear Fit")
plt.xlabel("Frame Index")
plt.ylabel("Distance (m)")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.savefig(output_path)
plt.close()

print(f"✅ Scene {scene_id} に RANSAC を適用し保存しました: {output_path}")


✅ Scene 047 に RANSAC を適用し保存しました: ./scene211_ransac_fixed.png
